In [1]:
pip install transformers sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 92.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [3]:
knowledge_base = [
    "The capital of France is Paris.",
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.",
    "Python is a high-level, general-purpose programming language.",
    "Large Language Models are a type of artificial intelligence model.",
    "Hugging Face is a company that provides tools for building AI applications."
]

In [4]:
from sentence_transformers import SentenceTransformer
import faiss

# Load a pre-trained Sentence Transformer model
embedding_model = SentenceTransformer('all-mpnet-base-v2')

# Generate embeddings for the knowledge base
embeddings = embedding_model.encode(knowledge_base)

# Build an index for efficient similarity search using FAISS
embedding_dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dimension)  # Using L2 distance for similarity
index.add(embeddings)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
def retrieve_relevant_documents(query, index, knowledge_base, embedding_model, top_k=2):
    """Retrieves the top_k most relevant documents from the knowledge base for a given query."""
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    relevant_documents = [knowledge_base[i] for i in indices[0]]
    return relevant_documents

In [6]:
from transformers import pipeline

# Load a pre-trained text generation model (you might need to experiment with different models)
llm = pipeline("text-generation", model="gpt2")

def generate_answer_with_context(query, retrieved_documents, llm):
    """Generates an answer based on the query and retrieved documents."""
    context = "\n".join(retrieved_documents)
    prompt = f"Based on the following information: {context}\n\nAnswer the question: {query}"
    output = llm(prompt, max_length=200, num_return_sequences=1, pad_token_id=llm.tokenizer.eos_token_id)
    answer = output[0]['generated_text']
    return answer

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


In [8]:
if __name__ == "__main__":
    query = "What is the capital of France?"
    retrieved_docs = retrieve_relevant_documents(query, index, knowledge_base, embedding_model)
    print("Retrieved Documents:")
    for doc in retrieved_docs:
        print(f"- {doc}")

    answer = generate_answer_with_context(query, retrieved_docs, llm)
    print("\nGenerated Answer:")
    print(answer)

    query_python = "What is Python?"
    retrieved_docs_python = retrieve_relevant_documents(query_python, index, knowledge_base, embedding_model)
    print("\nRetrieved Documents for Python:")
    for doc in retrieved_docs_python:
        print(f"- {doc}")

    answer_python = generate_answer_with_context(query_python, retrieved_docs_python, llm)
    print("\nGenerated Answer for Python:")
    print(answer_python)

    query_ai = "Tell me about Large Language Models."
    retrieved_docs_ai = retrieve_relevant_documents(query_ai, index, knowledge_base, embedding_model)
    print("\nRetrieved Documents for LLMs:")
    for doc in retrieved_docs_ai:
        print(f"- {doc}")

    answer_ai = generate_answer_with_context(query_ai, retrieved_docs_ai, llm)
    print("\nGenerated Answer for LLMs:")
    print(answer_ai)

Retrieved Documents:
- The capital of France is Paris.
- The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.

Generated Answer:
Based on the following information: The capital of France is Paris.
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.

Answer the question: What is the capital of France? The answer is: The capital of the EU. See this page for other information about the European Union.

Why is France so important? French people are very active. Some 1.2 million people live on less than €300 per capita. Only around 70 per cent of the population are of EU descent.

Many French newspapers and TV stations (including those run by the French monarchy) cover the country's political history, even for a time, in the national papers, in the state-run media, and in the mainstream European newspapers and media organisations. In May, 2011, the FN launched an unprecedented appeal on Social (the party in general) (The